## Задание 1
Примените Post-Training Quantization (PTQ) к предварительно обученной BERT-модели для классификации, оцените влияние PTQ на скорость инференса, размер модели и поведение выходов (логитов). Следуйте алгоритму:

Запустите инференс FP32 n_runs раз и посчитайте среднее время одного прогона. До любой квантизации модель работает в FP32. На шаге 1 запускайте инференс без квантизации и измерьте время как baseline.

Примените динамическую квантизацию к слоям nn.Linear: `torch.quantization.quantize_dynamic`.

Повторите измерение времени для квантизованной модели.

Сравните логиты FP32 и квантизованной модели: выведите первые элементы, посчитайте L2-норму и разности max–abs.

Сравните размеры файлов (state_dict) FP32 и квантизованной модели.

In [9]:
import time
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

print(f"Supported quantization engines: {torch.backends.quantized.supported_engines}")
torch.backends.quantized.engine = torch.backends.quantized.supported_engines[0]
print(f"Current quantization engine: {torch.backends.quantized.engine}")

device = 'cpu'
model_name = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name, ignore_mismatches=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, ignore_mismatched_sizes=True)
model.eval()
model_cpu = model.to(device)

text = 'This is a sample sentence for quantization benchmark.'
inputs = tokenizer(text, return_tensors='pt')

n_runs = 50

with torch.inference_mode():
    # исходный инференс
    start = time.time()
    for _ in range(n_runs):
        _ = model_cpu(**inputs)
    t_orig = (time.time() - start) / n_runs
print(f'FP32 avg inference time (per run): {t_orig:.6f} s')

print('Применяем динамическую квантизацию...')
# Нужно применить quantize_dynamic к model_cpu (квантуем torch.nn.Linear)
quantized_model = torch.quantization.quantize_dynamic(model_cpu, {torch.nn.Linear}, dtype=torch.qint8)

quantized_model.eval()

with torch.inference_mode():
    start = time.time()
    for _ in range(n_runs):
        _ = quantized_model(**inputs)
    t_q = (time.time() - start) / n_runs
print(f'Quantized avg inference time (per run): {t_q:.6f} s')

with torch.inference_mode():
    logits_fp32 = model_cpu(**inputs).logits.detach()
    logits_q = quantized_model(**inputs).logits.detach()

print('Примеры логитов (FP32 vs Quantized):')
print(logits_fp32[0][:6].tolist())
print(logits_q[0][:6].tolist())

# Посчитать L2 и max-abs разницу между logits_fp32 и logits_q.
# Формула L2: l2 = ||logits_fp32 - logits_q||_2
# max_abs = max(abs(logits_fp32 - logits_q))
l2 = torch.norm(logits_fp32 - logits_q).item()
max_abs = (logits_fp32 - logits_q).abs().max()
print(f'L2 diff: {l2:.6f}, max abs diff: {max_abs:.6f}')

# Сохранение state_dict'ов и сравнение размеров
tmp_fp = 'model_fp32.pth'
tmp_q = 'model_q.pth'
torch.save(model_cpu.state_dict(), tmp_fp)
torch.save(quantized_model.state_dict(), tmp_q)
print('FP32 size (MB):', os.path.getsize(tmp_fp)/1024/1024)
print('Quant size (MB):', os.path.getsize(tmp_q)/1024/1024)

Supported quantization engines: ['qnnpack']
Current quantization engine: qnnpack


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11340.27it/s]


FP32 avg inference time (per run): 0.034518 s
Применяем динамическую квантизацию...


/var/folders/_4/p98h_g953379cg5tzm5_l5p40000gn/T/ipykernel_41052/451773457.py:35: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(model_cpu, {torch.nn.Linear}, dtype=torch.qint8)


Quantized avg inference time (per run): 0.026572 s
Примеры логитов (FP32 vs Quantized):
[0.7682597041130066, 0.1561252325773239]
[0.48848363757133484, 0.0005553454393520951]
L2 diff: 0.320120, max abs diff: 0.279776
FP32 size (MB): 417.723596572876
Quant size (MB): 173.08405017852783


## QAT Quant-Aware training

Чтобы при обучении модели имитировать квантизацию, нужно задать конфигурацию квантизации. Платформа (бэкенд) в PyTorch Quantization — это реализация ядра квантизованных операций, которая оптимизирована под конкретное «железо» и вычислительные библиотеки. PyTorch поддерживает несколько таких платформ:
- `fbgemm` — (Facebook General Matrix Multiplication) оптимизированный бэкенд для x86 CPU (Intel/AMD). Используется по умолчанию на серверных процессорах.
- `qnnpack` — оптимизированный бэкенд для ARM CPU (мобильные устройства).
- `onednn` — новый бэкенд для Intel CPU.

С помощью `get_default_qat_qconfig` (fbgemm) можно сказать PyTorch: «Подготовь модель к квантизации так, чтобы все fake-кванты и наблюдатели были совместимы с ядром fbgemm». Это влияет на то, какие квантизованные операции будут вставлены и как именно будут вычисляться INT8 в финальной модели.

Простой пример реализации QAT для BERT-модели:

In [10]:
import torch
from transformers import AutoModelForSequenceClassification
from torch.ao.quantization import get_default_qat_qconfig, prepare_qat, convert

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
model.train()

# Выбираем qconfig, ориентируясь на целевой бэкенд (например, 'fbgemm' для CPU)
qconfig = get_default_qat_qconfig("fbgemm")
model.qconfig = qconfig

# Вставляем fake-quant и observers
prepare_qat(model, inplace=True)

# Fine-tuning: делаем несколько эпох с маленьким lr, следим за валидацией
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
for epoch in range(1, 4):
    for batch in train_loader:
        # inputs - подготовленные батчи
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    # полезно иногда запускать eval на валидации, чтобы наблюдатели собирали статистику

# Перевод в режим eval и конверсия в истинную INT8 модель
model.eval()
model_int8 = convert(model.cpu())
# Далее model_int8 можно экспортировать или запускать в рантайме

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11161.48it/s]
/var/folders/_4/p98h_g953379cg5tzm5_l5p40000gn/T/ipykernel_41052/3177213749.py:13: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepare_qat(model, inplace=True)
/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 5/.venv312/lib/python3.12/site-packages/torch/ao/quantization/obs

NameError: name 'train_loader' is not defined